# 03. Competição entre os dois algoritmos

- **Modelo A, Ridge:** regressão linear com penalização L2. Simples, rápido, segura bem atributos parecidos entre si (os lags da temperatura).
- **Modelo B, HistGradientBoosting:** árvores em sequência, cada uma corrigindo o erro das anteriores. Pega relações não lineares, como o efeito da frente fria que só aparece quando a pressão cai e o vento vira para sul.

Os dois usam exatamente as features eleitas no notebook 02, o mesmo treino, o mesmo Pipeline de pré-processamento (imputação pela mediana e padronização), a mesma semente e a mesma janela de validação, com o MAE como métrica principal. A persistência (amanhã = hoje) entra como referência.

**Por que treinar só com o treino, sem juntar a validação depois?** O modelo que pode ser promovido é exatamente o que foi avaliado. E como o vigente nunca viu os dias de validação dos ciclos seguintes, a comparação com ele no notebook 04 continua justa.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src import ciclo, modelagem, preprocessamento

cfg = ciclo.carregar_config()
estado = ciclo.atual()
mlflow = ciclo.configurar_mlflow(cfg)
dataset, janelas = preprocessamento.carregar(cfg)
pd.set_option("display.precision", 3)
print(f"ciclo {estado['cycle_id']} | {estado['data_version']}")

ciclo 20261107-205440 | dv-20261024-622a2671


In [2]:
treino, validacao = janelas["treino"], janelas["validacao"]
eleitas = estado["votacao"]["features"]
semente = cfg["projeto"]["semente"]
persistencia = modelagem.persistencia(validacao)
datas = {"treino_inicio": str(treino.index.min().date()), "treino_fim": str(treino.index.max().date()),
         "validacao_inicio": str(validacao.index.min().date()), "validacao_fim": str(validacao.index.max().date())}

resultados = {}
for papel, spec in cfg["modelos"].items():
    modelo = modelagem.treinar(spec["algoritmo"], spec["params"], treino, eleitas, semente)
    m = modelagem.metricas(validacao.alvo, modelo.predict(validacao[eleitas]))
    tags = ciclo.tags(estado, "treino") | {"papel": papel, "algoritmo": spec["algoritmo"]}
    with mlflow.start_run(run_name=f"{estado['cycle_id']}-{papel}-{spec['algoritmo']}", tags=tags) as run:
        mlflow.log_params(spec["params"] | datas | {"algoritmo": spec["algoritmo"], "semente": semente,
                                                    "features": ",".join(eleitas), "n_features": len(eleitas)})
        mlflow.log_metrics({f"{nome}_validacao": valor for nome, valor in m.items()}
                           | {"mae_persistencia_validacao": persistencia["mae"]})
        info = mlflow.sklearn.log_model(modelo, name="modelo", input_example=validacao[eleitas].head(3),
                                        skops_trusted_types=modelagem.TIPOS_CONFIAVEIS)
    resultados[papel] = {"algoritmo": spec["algoritmo"], "run_id": run.info.run_id,
                         "model_uri": info.model_uri, "validacao": m}
    print(f"modelo {papel} ({spec['algoritmo']}): MAE {m['mae']:.3f} | run {run.info.run_id}")

modelo A (ridge): MAE 1.929 | run a0312efc00bd46399ae8177441e0bdb3


modelo B (hist_gradient_boosting): MAE 1.887 | run 63182e0a1f344b449fc6b3baa7e97bb7


In [3]:
tabela = pd.DataFrame({f"{p} ({r['algoritmo']})": r["validacao"] for p, r in resultados.items()}
                      | {"persistência": persistencia}).T
tabela

,mae,rmse,r2
A (ridge),1.929,2.438,0.707
B (hist_gradient_boosting),1.887,2.406,0.714
persistência,2.569,3.349,0.447


In [4]:
estado["competicao"] = resultados | {"persistencia": persistencia, "treino_ate": datas["treino_fim"],
                                     "janela_validacao": [datas["validacao_inicio"], datas["validacao_fim"]]}
ciclo.salvar(estado)